# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIRˆ² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is defined via a Croissant schema URL and contains multiple record sets and rich metadata for FAIR research.

In [ ]:
# Install mlcroissant if it is not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from Croissant using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset (schema + resources)
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # This is a mlcroissant.Metadata object

print(f"Dataset Title: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
This section explores available record sets and their fields, listed by their `@id`.

**Note:** You can browse all record sets, fields, and their croissant `@id`s through the metadata interface.

In [ ]:
# List all available record sets with their @id and display their fields' @ids

print("Available Record Sets (by @id):")
record_set_ids = []
for rs in meta.record_sets:
    print(f"- {rs.id}: {rs.name}")
    record_set_ids.append(rs.id)
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id} ({field.name})")

# Show columns for the first record set
if len(meta.record_sets) > 0:
    rs0 = meta.record_sets[0]
    print("\nColumns for first RecordSet:")
    for col in rs0.columns:
        print(f"- {col.id} ({col.name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the record set and field `@id`s from the overview, as required by Croissant and mlcroissant conventions.

In [ ]:
# Extract data from all record sets and map them by their @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with {len(df)} records and columns:")
    print(list(df.columns))
    print()

# Pick the first record set for demo
main_record_set_id = record_set_ids[0]
print(f"First 5 rows of RecordSet {main_record_set_id} (by @id):")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrates typical processing such as filtering, normalization, and grouping using fields and columns referenced by their `@id`.

Let's choose a numeric field from the main record set for analysis (e.g., protein abundance or molecular weight as suggested by the schema description).

In [ ]:
df = dataframes[main_record_set_id]

print("Numeric columns available for analysis:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"- {col}")

# For this demo, let's use 'abundance' or 'MW' if present
possible_fields = ['abundance', 'MW', 'coverage', 'peptide_count', 'number_peptides']
numeric_field = None
for fn in possible_fields:
    if fn in df.columns:
        numeric_field = fn
        break

if numeric_field is None:
    # Fall back to first numeric column
    num_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(num_cols) > 0:
        numeric_field = num_cols[0]
    else:
        raise ValueError("No numeric columns available for EDA.")

print(f"\nChoosing numeric field '@id': {numeric_field}")

threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0

# Filter records where the chosen numeric field > threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (Z-score)
filtered_df.loc[:, f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical column if available (e.g., 'sample', 'accession')
group_fields = ['sample', 'accession', 'protein_id', 'description']
group_field = None
for gn in group_fields:
    if gn in df.columns:
        group_field = gn
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field}, average of {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the top grouped values if grouping was possible.

*Requires matplotlib/pandas plotting.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping was possible, show top groups
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    topn = grouped_df.sort_values(by=numeric_field, ascending=False).head(10)
    sns.barplot(x=group_field, y=numeric_field, data=topn)
    plt.title(f"Top 10 {group_field}s by average {numeric_field}")
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion
- The FAIRˆ² dataset provides rich protein mass spectrometry data, annotated by Croissant `@id` for reproducibility.
- Using `mlcroissant`, you can robustly load, query, and process all record sets and fields by their Croissant `@id`.
- Exploration shows valuable numeric and categorical attributes suitable for downstream research.
- For full documentation, visit the [mlcroissant documentation](https://mlcommons.github.io/croissant/) or inspect the schema directly.